# Phase-0 — Ultrasound-Specific Foundation Model External Transfer Kill-Test v1.1

## Scientific question

> **Does ultrasound-specific pre-training provide an externally transferable advantage over an
architecture-matched generic visual foundation model in breast ultrasound?**

This is a kill-test. It is not a full benchmark and it is not designed to make URFM win.

---

## Primary matched comparison

### URFM ViT-B/16
Official ultrasound-specific checkpoint:

`QingboKang/URFM / mae_vit_base_patch16_dec768d8b_all_biomedclip_1199.pth`

- ViT-B/16 encoder
- representation-based masked image modeling
- >1M ultrasound images across 15 organs
- input size 224×224

### Generic ImageNet MAE ViT-B/16
`timm/vit_base_patch16_224.mae`

- same ViT-B/16 architecture
- MAE pre-training on natural images
- 224×224
- same frozen CLS-token representation protocol used in this experiment

**Primary scientific contrast = URFM − ImageNet-MAE.**

The two models use the same:
- input size;
- square-resize preprocessing;
- ImageNet normalization;
- CLS-token pooling;
- frozen-backbone protocol;
- linear-head optimizer/training budget;
- source splits.

---

## Secondary modern generic control

### DINOv2 ViT-B/14
Existing completed **native 518×518** feature cache is reused.

DINOv2 is not architecture- or resolution-matched to URFM, so it is **contextual evidence**, not the
primary causal contrast about pre-training domain.

---

## Pre-training provenance boundary

Published URFM pre-training sources for the breast subset are:
- BUV;
- ultrasoundcases.info;
- private breast ultrasound data.

BUS-BRA, BrEaST and BUSI are not listed as named public pre-training sources in the URFM paper.
However, overlap with the private pre-training component cannot be independently ruled out.

Therefore:
- URFM vs MAE is described as a **declared-provenance-aware** comparison;
- we do not claim mathematically proven zero pre-training overlap;
- DINOv2 is secondary because its web-scale pre-training provenance cannot be audited at the same
dataset-by-dataset level.

---

# Datasets and source directions

Primary source datasets have patient/case identifiers.

## Source A — BUS-BRA
- 1,064 patients
- pathology-stratified 80/20 patient split
- source split seed 42
- multiple images are averaged to one patient representation before linear-head fitting

External targets:
- BrEaST: 252 lesion cases
- BUSI: 647 benign/malignant lesion images

## Source B — BrEaST
- 252 lesion cases
- pathology-stratified 80/20 case split
- source split seed 42

External targets:
- BUS-BRA: 1,064 patients
- BUSI: 647 benign/malignant lesion images

BUSI is external-only because reliable patient identifiers are unavailable.

Thus the Phase-0 primary external matrix contains:

\[
2\ sources \times 2\ unseen\ targets = 4\ scenarios.
\]

---

# Adaptation

Frozen backbone + linear head only.

For every representation × source:
- head seeds: 42, 43, 44
- AdamW
- lr = 1e-4
- weight decay = 1e-2
- batch = 16
- class-weighted CE
- 30 epochs
- best checkpoint = highest source-validation AUROC
- no scheduler
- no target tuning
- no image augmentation after feature caching

No LoRA, VPT or full fine-tuning in Phase-0.

---

# Primary statistics

For each source→target scenario, compare URFM and matched ImageNet-MAE.

For each pathology-stratified target bootstrap:
1. use the same resampled target units for both models;
2. compute AUROC for each of the three head seeds;
3. compute seed-balanced effect

\[
\Delta_{scenario}
=
\frac{1}{3}\sum_s
\left(AUC_{URFM,s}-AUC_{MAE,s}\right).
\]

3000 bootstrap replicates.

DINOv2 results are reported descriptively.

---

# Frozen decision rules

There are two scientifically interesting GO paths.

## GO-A — `GO_ULTRASOUND_PRETRAINING_EXTERNAL_ADVANTAGE`

All:
1. URFM−MAE external AUROC > 0 in at least **3/4** scenarios;
2. median scenario effect ≥ **+0.03 AUROC**;
3. at least **3/4** scenarios have bootstrap `P(URFM > MAE) ≥ 0.90`.

Interpretation:
> ultrasound-specific pre-training shows a reproducible externally transferable advantage over the
architecture-matched generic MAE representation.

## GO-B — `GO_SOURCE_ADVANTAGE_FAILS_TO_TRANSFER`

All:
1. URFM−MAE source-validation AUROC ≥ **+0.02** for **both** source datasets;
2. median external URFM−MAE effect ≤ **+0.01**;
3. source-to-external attenuation ≥ **0.02 AUROC** in at least **3/4** scenarios;
4. URFM has bootstrap `P(URFM > MAE) ≥ 0.90` in at most **1/4** external scenarios.

Interpretation:
> a source-domain advantage of ultrasound-specific pre-training does not reliably translate into
cross-dataset external superiority.

## STOP — `STOP_FM_EXTERNAL_TRANSFER_DIRECTION`

If neither GO-A nor GO-B passes.

After STOP:
- no LoRA;
- no VPT;
- no full fine-tuning;
- no extra seed rescue;
- no new ultrasound FM added to rescue the direction.

This is the final breast-US Phase-0 kill rule.


### v1.1 access amendment

URFM's official Hugging Face repository is gated. v1.1 adds an explicit gated-access step,
authentication check, and a persistent Drive checkpoint fallback. No scientific protocol,
dataset, model comparison, preprocessing, seed, threshold, bootstrap, or GO/STOP criterion changed.


In [ ]:
!pip -q install timm==1.0.28 huggingface_hub kagglehub scikit-learn pandas scipy pillow tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, re, gc, json, math, random, zipfile
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import timm
import kagglehub
from huggingface_hub import hf_hub_download, login, whoami

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

ROOT = Path('/content/drive/MyDrive/SPATIOSPECTRAL_PEFT_PILOT')

# Existing completed project artifacts.
D1 = ROOT/'linear_readout_control_v1_0'
D1O = D1/'outputs'

PACKA = ROOT/'reviewer_proof_pack_A_v1_0'
PACKAO = PACKA/'outputs'
PACKAP = PACKA/'progress'

# New Phase-0 artifacts.
EXP = ROOT/'fm_external_transfer_phase0_v1_0'
OUT = EXP/'outputs'
PROG = EXP/'progress'
CKPT = EXP/'checkpoints'
for p in [EXP, OUT, PROG, CKPT]:
    p.mkdir(parents=True, exist_ok=True)

URFM_REPO_ID = 'QingboKang/URFM'
URFM_FILENAME = 'mae_vit_base_patch16_dec768d8b_all_biomedclip_1199.pth'

URFM_LOCAL_DIR = ROOT/'external_models'/'URFM'
URFM_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
URFM_LOCAL_PATH = URFM_LOCAL_DIR/URFM_FILENAME

MAE_TIMM_MODEL = 'vit_base_patch16_224.mae'
DINO2_REP = 'DINOv2-518'

REPRESENTATIONS = ['URFM-224', 'ImageNet-MAE-224', DINO2_REP]
PRIMARY_MODELS = ['URFM-224', 'ImageNet-MAE-224']

SOURCE_SPLIT_SEED = 42
SOURCE_VAL_FRAC = 0.20
HEAD_SEEDS = [42,43,44]

IMAGE_SIZE_MATCHED = 224
FEATURE_BATCH = 8

HEAD_BATCH = 16
HEAD_EPOCHS = 30
HEAD_LR = 1e-4
HEAD_WEIGHT_DECAY = 1e-2

BOOTSTRAP_REPS = 3000
BOOTSTRAP_SEED = 91001

# GO-A.
GOA_MIN_POSITIVE_SCENARIOS = 3
GOA_MEDIAN_EFFECT = 0.03
GOA_MIN_BOOT_SUPPORTED = 3
GOA_BOOT_PROB = 0.90

# GO-B.
GOB_SOURCE_ADVANTAGE = 0.02
GOB_MAX_EXTERNAL_MEDIAN = 0.01
GOB_ATTENUATION = 0.02
GOB_MIN_ATTENUATED_SCENARIOS = 3
GOB_MAX_BOOT_SUPPORTED_EXTERNAL = 1

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Feature-extraction device:", DEVICE)
if DEVICE.type == 'cuda':
    print(torch.cuda.get_device_name(0))

CONFIG = {
    'experiment':'Ultrasound-specific FM external transfer Phase-0 v1.0',
    'primary_comparison':'URFM-224 vs ImageNet-MAE-224',
    'secondary_control':'DINOv2-518',
    'sources':['BUSBRA','BREAST'],
    'external_targets_by_source':{
        'BUSBRA':['BREAST','BUSI'],
        'BREAST':['BUSBRA','BUSI'],
    },
    'matched_input_size':224,
    'matched_pooling':'CLS token',
    'matched_normalization':'ImageNet mean/std',
    'source_split_seed':SOURCE_SPLIT_SEED,
    'head_seeds':HEAD_SEEDS,
    'head_epochs':HEAD_EPOCHS,
    'bootstrap_reps':BOOTSTRAP_REPS,
    'target_tuning':False,
    'decision_paths':[
        'GO_ULTRASOUND_PRETRAINING_EXTERNAL_ADVANTAGE',
        'GO_SOURCE_ADVANTAGE_FAILS_TO_TRANSFER',
        'STOP_FM_EXTERNAL_TRANSFER_DIRECTION',
    ],
}
(EXP/'FROZEN_PHASE0_CONFIG.json').write_text(
    json.dumps(CONFIG,indent=2),encoding='utf-8'
)

PROVENANCE = {
    'URFM':{
        'official_checkpoint':f'{URFM_REPO_ID}/{URFM_FILENAME}',
        'declared_pretraining_images':1003465,
        'declared_breast_sources':['BUV','ultrasoundcases.info','private'],
        'evaluation_public_datasets_named_as_pretraining_sources':[],
        'boundary':(
            'Named public pretraining sources do not list BUS-BRA, BrEaST or BUSI; '
            'overlap with private data cannot be independently excluded.'
        ),
    },
    'ImageNet_MAE':{
        'model':MAE_TIMM_MODEL,
        'pretraining_domain':'ImageNet natural images',
        'boundary':'Architecture-matched primary generic visual SSL baseline.',
    },
    'DINOv2':{
        'representation':'existing native-518 cache',
        'boundary':(
            'Secondary contextual control only; not architecture/resolution matched and '
            'dataset-level pretraining provenance is not audited here.'
        ),
    },
}
(EXP/'PRETRAINING_PROVENANCE_AUDIT.json').write_text(
    json.dumps(PROVENANCE,indent=2),encoding='utf-8'
)


## 1. Utilities

In [ ]:
IMAGENET_MEAN = torch.tensor(
    [0.485,0.456,0.406],dtype=torch.float32
).view(3,1,1)
IMAGENET_STD = torch.tensor(
    [0.229,0.224,0.225],dtype=torch.float32
).view(3,1,1)

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def atomic_csv(df,path):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+'.tmp')
    df.to_csv(tmp,index=False)
    os.replace(tmp,path)

def atomic_json(obj,path):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2),encoding='utf-8')
    os.replace(tmp,path)

def canonical_patient_key(v):
    s=str(v).strip()
    if re.fullmatch(r'\d+(?:\.0+)?',s):
        return str(int(float(s)))
    return s.lower()

def pil_tensor(img,size=224):
    img=img.convert('RGB').resize(
        (size,size),Image.Resampling.BILINEAR
    )
    x=torch.from_numpy(
        np.asarray(img,dtype=np.float32)/255.0
    ).permute(2,0,1)
    return (x-IMAGENET_MEAN)/IMAGENET_STD

def malignant_margin(logits):
    return logits[:,1]-logits[:,0]

def auc_ap(y,s):
    y=np.asarray(y,dtype=int)
    s=np.asarray(s,dtype=float)
    return (
        float(roc_auc_score(y,s)),
        float(average_precision_score(y,s))
    )

def all_exist(values):
    return bool(np.all([Path(str(x)).exists() for x in values]))


## 2. Load / canonicalize dataset manifests

Scientific identities, labels and source splits are not reconstructed from filenames.

Only stale filesystem paths are remapped if necessary.


In [ ]:
busi_path_candidates = [
    PACKAO/'BUSI_RUNTIME_REMAPPED_MANIFEST.csv',
    D1O/'BUSI_LINEAR_CONTROL_MANIFEST.csv',
]
busi_path = next(
    (p for p in busi_path_candidates if p.exists()),
    None
)
if busi_path is None:
    raise FileNotFoundError("BUSI project manifest not found.")

busbra_path_candidates = [
    PACKAO/'BUSBRA_RUNTIME_REMAPPED_MANIFEST.csv',
    D1O/'BUSBRA_EXTERNAL_MANIFEST.csv',
]
busbra_path = next(
    (p for p in busbra_path_candidates if p.exists()),
    None
)
if busbra_path is None:
    raise FileNotFoundError("BUS-BRA project manifest not found.")

breast_path = PACKAO/'BREAST_LESION_MANIFEST.csv'
if not breast_path.exists():
    raise FileNotFoundError(
        "BrEaST Pack-A lesion manifest missing: "+str(breast_path)
    )

busi=pd.read_csv(busi_path)
busbra=pd.read_csv(busbra_path)
breast=pd.read_csv(breast_path)

# ---- BUSI runtime path repair if needed ----
if not all_exist(busi.image_path):
    print("Remapping stale BUSI runtime paths with KaggleHub.")
    root=Path(
        kagglehub.dataset_download(
            'aryashah2k/breast-ultrasound-images-dataset'
        )
    )
    index={}
    for p in root.rglob('*.png'):
        if '_mask' in p.stem.lower():
            continue
        index[(p.parent.name.strip().lower(),p.stem)] = str(p)

    resolved=[]
    missing=[]
    for _,r in busi.iterrows():
        key=(str(r.label_text).strip().lower(),str(r.image_id).strip())
        p=index.get(key)
        if p is None:
            missing.append(key)
        resolved.append(p)

    if missing:
        raise RuntimeError(
            f"Could not resolve BUSI examples: {missing[:10]}"
        )
    busi=busi.copy()
    busi['image_path']=resolved
    atomic_csv(busi,OUT/'BUSI_PHASE0_RUNTIME_MANIFEST.csv')

# ---- BUS-BRA runtime path repair if needed ----
if not all_exist(busbra.image_path):
    candidates=[
        ROOT/'data'/'BUSBRA_OFFICIAL'/'BUSBRA'/'Images',
        ROOT/'data'/'BUS-BRA'/'Images',
    ]
    image_dir=next((p for p in candidates if p.exists()),None)
    if image_dir is None:
        raise FileNotFoundError(
            "BUS-BRA raw image directory not found in known project paths."
        )
    remapped=[
        str(image_dir/f"{str(iid)}.png")
        for iid in busbra.image_id
    ]
    if not all_exist(remapped):
        bad=[x for x in remapped if not Path(x).exists()][:10]
        raise RuntimeError(f"BUS-BRA remap failed: {bad}")
    busbra=busbra.copy()
    busbra['image_path']=remapped
    atomic_csv(busbra,OUT/'BUSBRA_PHASE0_RUNTIME_MANIFEST.csv')

if not all_exist(breast.image_path):
    raise FileNotFoundError(
        "BrEaST manifest image paths are stale. Keep the extracted Pack-A BrEaST folder on Drive."
    )

# Frozen lesion-only BUSI cohort.
if 'split' in busi.columns:
    busi_train=busi[busi.split=='train'].reset_index(drop=True)
    busi_val=busi[busi.split=='val'].reset_index(drop=True)
    busi_all=pd.concat([busi_train,busi_val],ignore_index=True)
else:
    busi_all=busi.reset_index(drop=True)

busbra['patient_key']=busbra.patient_key.map(canonical_patient_key)
breast['patient_key']=breast.patient_key.astype(str)

assert len(busi_all)==647
assert int((busi_all.y==0).sum())==437
assert int((busi_all.y==1).sum())==210

assert len(busbra)==1875
assert busbra.patient_key.nunique()==1064

assert len(breast)==252
assert breast.patient_key.nunique()==252
assert int((breast.y==0).sum())==154
assert int((breast.y==1).sum())==98

print(
    "PASS manifests:",
    "BUSI",len(busi_all),
    "| BUS-BRA",len(busbra),"images /",busbra.patient_key.nunique(),"patients",
    "| BrEaST",len(breast),"cases"
)


## 2.5 URFM gated-access authentication

The official URFM Hugging Face repository is gated.

Before this notebook can download the checkpoint:

1. Open `https://huggingface.co/QingboKang/URFM` in a browser while logged into your Hugging Face
   account.
2. Request/accept access to the gated repository.
3. If the repository uses manual approval, wait until the model authors approve the request.
4. Authenticate this Colab runtime with the **same Hugging Face account**.

This cell first checks whether the checkpoint has already been placed manually in Drive:

`MyDrive/SPATIOSPECTRAL_PEFT_PILOT/external_models/URFM/`

If not, it runs Hugging Face notebook login and verifies access before feature extraction.

Do not paste a token into notebook source code that you plan to share.


In [ ]:
def resolve_urfm_checkpoint():
    # Option A: user downloaded the official gated checkpoint manually
    # and placed it in Drive. This avoids storing any HF token in notebook code.
    if URFM_LOCAL_PATH.exists():
        print("Using URFM checkpoint from Drive:", URFM_LOCAL_PATH)
        return str(URFM_LOCAL_PATH)

    print(
        "URFM checkpoint is not present in Drive.\n"
        "The official Hugging Face repository is gated.\n"
        "Make sure access has first been granted in the browser for:\n"
        "https://huggingface.co/QingboKang/URFM"
    )

    # Interactive notebook authentication.
    # login() uses Hugging Face's secure notebook authentication flow.
    try:
        login()
    except Exception as e:
        raise RuntimeError(
            "Hugging Face authentication did not complete. "
            "Request URFM repository access in the browser first, then rerun this cell."
        ) from e

    try:
        ident = whoami()
        print("Authenticated Hugging Face user:", ident.get("name", "<unknown>"))
    except Exception:
        pass

    try:
        p = hf_hub_download(
            repo_id=URFM_REPO_ID,
            filename=URFM_FILENAME,
        )
    except Exception as e:
        msg = str(e)
        if (
            "401" in msg
            or "GatedRepoError" in type(e).__name__
            or "restricted" in msg.lower()
            or "gated" in msg.lower()
        ):
            raise RuntimeError(
                "Authentication succeeded but this Hugging Face account does not yet "
                "have access to the gated URFM repository. "
                "Open https://huggingface.co/QingboKang/URFM in the browser, "
                "request access, and wait for approval if the authors use manual gating. "
                "Then rerun this cell. No training or feature extraction needs to be repeated."
            ) from e
        raise

    # Make a persistent Drive copy so future Colab runtimes do not need to
    # download the gated checkpoint again.
    import shutil
    shutil.copy2(p, URFM_LOCAL_PATH)
    print("URFM checkpoint cached to Drive:", URFM_LOCAL_PATH)
    return str(URFM_LOCAL_PATH)

URFM_RESOLVED_CHECKPOINT = resolve_urfm_checkpoint()


## 3. Build the primary matched frozen encoders

### URFM loader

The official checkpoint is loaded into a standard ViT-B/16 encoder.
Decoder and mask-token weights are ignored because Phase-0 uses only the encoder.

Critical encoder tensors must load:
- patch embedding;
- CLS token;
- positional embedding;
- first and last Transformer blocks;
- final encoder norm.

### ImageNet MAE

Loaded directly from timm as `vit_base_patch16_224.mae`.

Both output 768-dimensional CLS-token features.


In [ ]:
def build_urfm():
    ckpt_path = URFM_RESOLVED_CHECKPOINT
    print("URFM checkpoint:", ckpt_path)

    ck=torch.load(
        ckpt_path,map_location='cpu',weights_only=False
    )
    state=ck.get('state_dict',ck.get('model',ck))
    state={
        k.replace('module.',''):v
        for k,v in state.items()
    }

    model=timm.create_model(
        'vit_base_patch16_224',
        pretrained=False,
        num_classes=0,
        global_pool='token',
    )

    target=model.state_dict()
    filtered={}
    shape_mismatch=[]

    for k,v in state.items():
        if k.startswith('decoder_') or k=='mask_token':
            continue
        if k in target:
            if tuple(v.shape)==tuple(target[k].shape):
                filtered[k]=v
            else:
                shape_mismatch.append(
                    (k,tuple(v.shape),tuple(target[k].shape))
                )

    msg=model.load_state_dict(filtered,strict=False)

    critical=[
        'patch_embed.proj.weight',
        'cls_token',
        'pos_embed',
        'blocks.0.attn.qkv.weight',
        'blocks.11.attn.qkv.weight',
        'norm.weight',
        'norm.bias',
    ]
    absent=[k for k in critical if k not in filtered]
    if absent:
        raise RuntimeError(
            f"URFM critical encoder keys were not loaded: {absent}"
        )

    audit={
        'checkpoint':str(ckpt_path),
        'checkpoint_top_level_keys':list(ck.keys()) if isinstance(ck,dict) else [],
        'loaded_encoder_tensors':len(filtered),
        'missing_keys':list(msg.missing_keys),
        'unexpected_keys':list(msg.unexpected_keys),
        'shape_mismatch':shape_mismatch,
        'critical_keys_loaded':critical,
        'feature_protocol':'ViT-B/16 CLS token, 224x224',
    }
    atomic_json(audit,OUT/'URFM_LOAD_AUDIT.json')

    for p in model.parameters():
        p.requires_grad=False
    model=model.to(DEVICE).eval()

    with torch.no_grad():
        z=model(torch.zeros(1,3,224,224,device=DEVICE))
    if tuple(z.shape)!=(1,768):
        raise RuntimeError(f"Unexpected URFM feature shape {tuple(z.shape)}")

    print("PASS URFM encoder load:",audit)
    return model

def build_mae():
    try:
        model=timm.create_model(
            MAE_TIMM_MODEL,
            pretrained=True,
            num_classes=0,
            global_pool='token',
        )
    except Exception:
        model=timm.create_model(
            f'hf_hub:timm/{MAE_TIMM_MODEL}',
            pretrained=True,
            num_classes=0,
            global_pool='token',
        )

    for p in model.parameters():
        p.requires_grad=False
    model=model.to(DEVICE).eval()

    with torch.no_grad():
        z=model(torch.zeros(1,3,224,224,device=DEVICE))
    if tuple(z.shape)!=(1,768):
        raise RuntimeError(f"Unexpected MAE feature shape {tuple(z.shape)}")

    atomic_json({
        'model':MAE_TIMM_MODEL,
        'feature_protocol':'ViT-B/16 CLS token, 224x224',
        'feature_dim':768,
    },OUT/'MAE_LOAD_AUDIT.json')

    print("PASS ImageNet-MAE load.")
    return model


## 4. Interruption-safe feature extraction for URFM and MAE

In [ ]:
class ImageRows(Dataset):
    def __init__(self,rows):
        self.rows=rows.reset_index(drop=True)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self,i):
        r=self.rows.iloc[i]
        img=Image.open(r.image_path).convert('RGB')
        return {
            'x':pil_tensor(img,IMAGE_SIZE_MATCHED),
            'idx':torch.tensor(i,dtype=torch.long),
        }

def cache_paths(rep,dataset):
    tag=f"{rep.replace('-','_')}_{dataset}"
    return {
        'feat':PROG/f'{tag}_features.npy',
        'done':PROG/f'{tag}_done.npy',
        'status':PROG/f'{tag}_status.json',
    }

def open_mm(path,dtype,shape):
    path=Path(path)
    if not path.exists():
        mm=np.lib.format.open_memmap(
            path,mode='w+',dtype=dtype,shape=shape
        )
        mm[:]=0
        mm.flush()
        del mm
    mm=np.load(path,mmap_mode='r+')
    if mm.shape!=shape:
        raise RuntimeError(
            f"{path.name}: {mm.shape} != expected {shape}"
        )
    return mm

@torch.no_grad()
def extract(model,rows,rep,dataset):
    ds=ImageRows(rows)
    n=len(ds)
    p=cache_paths(rep,dataset)

    feat=open_mm(p['feat'],np.float32,(n,768))
    done=open_mm(p['done'],np.bool_,(n,))

    remaining=np.flatnonzero(
        ~np.asarray(done,dtype=bool)
    ).tolist()

    print(rep,dataset,":",n-len(remaining),"/",n,"complete")

    if remaining:
        if DEVICE.type!='cuda':
            raise RuntimeError(
                f"GPU required for unfinished feature extraction: {rep}/{dataset}"
            )

        loader=DataLoader(
            Subset(ds,remaining),
            batch_size=FEATURE_BATCH,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
        )

        model.eval()
        for b in tqdm(loader,desc=f'{rep}-{dataset}'):
            idx=b['idx'].numpy().astype(int)
            x=b['x'].to(DEVICE,non_blocking=True)
            z=model(x)
            if tuple(z.shape[1:])!=(768,):
                raise RuntimeError(
                    f"{rep}/{dataset}: feature shape {tuple(z.shape)}"
                )
            feat[idx]=z.detach().cpu().numpy().astype(np.float32)
            done[idx]=True
            feat.flush()
            done.flush()

        atomic_json({
            'representation':rep,
            'dataset':dataset,
            'complete':bool(np.asarray(done).all()),
            'n':n,
        },p['status'])

    if not bool(np.asarray(done).all()):
        raise RuntimeError(f"Incomplete feature cache: {rep}/{dataset}")

    out=np.asarray(feat,dtype=np.float32).copy()
    del feat,done
    return out

dataset_rows={
    'BUSI':busi_all,
    'BUSBRA':busbra,
    'BREAST':breast,
}

features={}

# URFM.
m=build_urfm()
features['URFM-224']={}
for d,rows in dataset_rows.items():
    features['URFM-224'][d]=extract(
        m,rows,'URFM-224',d
    )
m.to('cpu'); del m; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

# Architecture-matched ImageNet MAE.
m=build_mae()
features['ImageNet-MAE-224']={}
for d,rows in dataset_rows.items():
    features['ImageNet-MAE-224'][d]=extract(
        m,rows,'ImageNet-MAE-224',d
    )
m.to('cpu'); del m; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

print("PASS primary matched feature extraction.")


## 5. Reuse the completed DINOv2 native-518 cache

No DINOv2 backbone inference is repeated.


In [ ]:
d2_required={
    'BUSI_train':PACKAP/'D2_518_train_features.npy',
    'BUSI_val':PACKAP/'D2_518_val_features.npy',
    'BUSBRA':PACKAP/'D2_518_BUSBRA_features.npy',
    'BREAST':PACKAP/'D2_518_BREAST_features.npy',
}
missing={
    k:str(v) for k,v in d2_required.items()
    if not v.exists()
}
if missing:
    raise FileNotFoundError(
        "Completed DINOv2-518 cache missing; do not recompute automatically:\n"
        +json.dumps(missing,indent=2)
    )

features[DINO2_REP]={
    'BUSI':np.concatenate([
        np.asarray(
            np.load(d2_required['BUSI_train'],mmap_mode='r'),
            dtype=np.float32
        ),
        np.asarray(
            np.load(d2_required['BUSI_val'],mmap_mode='r'),
            dtype=np.float32
        ),
    ],axis=0),
    'BUSBRA':np.asarray(
        np.load(d2_required['BUSBRA'],mmap_mode='r'),
        dtype=np.float32
    ),
    'BREAST':np.asarray(
        np.load(d2_required['BREAST'],mmap_mode='r'),
        dtype=np.float32
    ),
}

for rep in REPRESENTATIONS:
    assert features[rep]['BUSI'].shape==(647,768)
    assert features[rep]['BUSBRA'].shape==(1875,768)
    assert features[rep]['BREAST'].shape==(252,768)

print("PASS DINOv2-518 reuse and feature-count audit.")


## 6. Build source statistical units

BUS-BRA:
- average image features within patient.

BrEaST:
- one case/image per patient.

BUSI:
- external-only image-level cohort.


In [ ]:
def patient_mean_features(X,manifest):
    groups=manifest.groupby(
        ['patient_key','y'],
        sort=True
    ).indices

    rows=[]
    vecs=[]
    for (pid,y),idx in groups.items():
        idx=np.asarray(idx,dtype=int)
        rows.append({
            'patient_key':str(pid),
            'y':int(y),
            'n_images':int(len(idx)),
        })
        vecs.append(
            np.asarray(X[idx],dtype=np.float32).mean(axis=0)
        )
    return pd.DataFrame(rows),np.stack(vecs).astype(np.float32)

source_units={}

for rep in REPRESENTATIONS:
    meta_bra,X_bra=patient_mean_features(
        features[rep]['BUSBRA'],busbra
    )

    source_units[(rep,'BUSBRA')]={
        'meta':meta_bra,
        'X':X_bra,
        'y':meta_bra.y.to_numpy(dtype=int),
    }

    meta_br=breast[
        ['patient_key','y']
    ].copy().reset_index(drop=True)

    source_units[(rep,'BREAST')]={
        'meta':meta_br,
        'X':features[rep]['BREAST'].copy(),
        'y':meta_br.y.to_numpy(dtype=int),
    }

# Verify identical patient/case identity across representations.
for source in ['BUSBRA','BREAST']:
    ref=source_units[(REPRESENTATIONS[0],source)]['meta']
    for rep in REPRESENTATIONS[1:]:
        cur=source_units[(rep,source)]['meta']
        if not np.array_equal(
            ref.patient_key.astype(str).to_numpy(),
            cur.patient_key.astype(str).to_numpy()
        ):
            raise RuntimeError(
                f"{source}: identity mismatch across representations."
            )
        if not np.array_equal(
            ref.y.to_numpy(dtype=int),
            cur.y.to_numpy(dtype=int)
        ):
            raise RuntimeError(
                f"{source}: label mismatch across representations."
            )

assert len(source_units[('URFM-224','BUSBRA')]['y'])==1064
assert len(source_units[('URFM-224','BREAST')]['y'])==252

print("PASS source-unit construction.")


## 7. Freeze source train/validation splits

In [ ]:
splits={}

for source in ['BUSBRA','BREAST']:
    y=source_units[('URFM-224',source)]['y']

    sss=StratifiedShuffleSplit(
        n_splits=1,
        test_size=SOURCE_VAL_FRAC,
        random_state=SOURCE_SPLIT_SEED,
    )
    tr,va=next(
        sss.split(np.zeros(len(y)),y)
    )
    splits[source]=(tr,va)

    meta=source_units[('URFM-224',source)]['meta'].copy()
    meta['source_split']='train'
    meta.loc[va,'source_split']='val'

    assert set(meta.iloc[tr].patient_key).isdisjoint(
        set(meta.iloc[va].patient_key)
    )

    atomic_csv(
        meta,
        OUT/f'{source}_PHASE0_SOURCE_SPLIT.csv'
    )

    print(
        source,
        "train",len(tr),
        "val",len(va),
        "| train malignant",int((y[tr]==1).sum()),
        "| val malignant",int((y[va]==1).sum())
    )


## 8. Train frozen-representation linear heads

Three independent head seeds per representation × source.

Backbone features remain fixed.


In [ ]:
class FeatDS(Dataset):
    def __init__(self,X,y):
        self.X=np.asarray(X,dtype=np.float32)
        self.y=np.asarray(y,dtype=np.int64)
    def __len__(self):
        return len(self.y)
    def __getitem__(self,i):
        return (
            torch.from_numpy(self.X[i]),
            torch.tensor(int(self.y[i]),dtype=torch.long)
        )

def weights_for(y):
    c=np.bincount(y,minlength=2)
    w=c.sum()/(2*np.maximum(c,1))
    w=w/w.mean()
    return torch.tensor(w,dtype=torch.float32)

@torch.no_grad()
def head_scores(head,X):
    head.eval()
    X=np.asarray(X,dtype=np.float32)
    loader=DataLoader(
        torch.from_numpy(X),
        batch_size=256,
        shuffle=False,
        num_workers=0,
    )
    out=[]
    for f in loader:
        logits=head(f)
        out.extend(
            malignant_margin(logits).numpy().tolist()
        )
    return np.asarray(out,dtype=float)

def train_head(rep,source,seed):
    seed_everything(seed)

    tag=f"{rep.replace('-','_')}_{source}_seed{seed}"
    best_path=CKPT/f'{tag}_best.pt'
    hist_path=OUT/f'{tag}_history.csv'
    status_path=OUT/f'{tag}_status.json'

    if best_path.exists() and status_path.exists():
        st=json.loads(status_path.read_text(encoding='utf-8'))
        if st.get('complete_30_epochs',False):
            h=nn.Linear(768,2)
            ck=torch.load(
                best_path,map_location='cpu',weights_only=False
            )
            h.load_state_dict(ck['state_dict'])
            h.eval()
            return h,st,pd.read_csv(hist_path)

    X=source_units[(rep,source)]['X']
    y=source_units[(rep,source)]['y']
    tr,va=splits[source]

    Xtr,Xva=X[tr],X[va]
    ytr,yva=y[tr],y[va]

    gen=torch.Generator()
    gen.manual_seed(seed)

    loader=DataLoader(
        FeatDS(Xtr,ytr),
        batch_size=HEAD_BATCH,
        shuffle=True,
        num_workers=0,
        generator=gen,
    )

    h=nn.Linear(768,2)
    opt=torch.optim.AdamW(
        h.parameters(),
        lr=HEAD_LR,
        weight_decay=HEAD_WEIGHT_DECAY,
    )
    cw=weights_for(ytr)

    best_auc=-np.inf
    best_epoch=None
    hist=[]

    for epoch in range(1,HEAD_EPOCHS+1):
        h.train()
        losses=[]

        for f,yy in loader:
            opt.zero_grad(set_to_none=True)
            logits=h(f)
            loss=F.cross_entropy(
                logits,yy,weight=cw
            )
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))

        s=head_scores(h,Xva)
        auc,ap=auc_ap(yva,s)

        hist.append({
            'representation':rep,
            'source':source,
            'seed':seed,
            'epoch':epoch,
            'train_loss':float(np.mean(losses)),
            'source_val_auc':auc,
            'source_val_auprc':ap,
        })

        if auc>best_auc:
            best_auc=auc
            best_epoch=epoch
            torch.save({
                'representation':rep,
                'source':source,
                'seed':seed,
                'epoch':epoch,
                'best_source_val_auc':best_auc,
                'state_dict':{
                    k:v.detach().cpu().clone()
                    for k,v in h.state_dict().items()
                }
            },best_path)

    hist=pd.DataFrame(hist)
    atomic_csv(hist,hist_path)

    ck=torch.load(
        best_path,map_location='cpu',weights_only=False
    )
    h.load_state_dict(ck['state_dict'])
    h.eval()

    st={
        'representation':rep,
        'source':source,
        'seed':seed,
        'complete_30_epochs':True,
        'best_epoch':int(best_epoch),
        'best_source_val_auc':float(best_auc),
        'trainable_params':1538,
        'feature_protocol':'frozen representation',
    }
    atomic_json(st,status_path)
    return h,st,hist

heads={}
status_rows=[]

for rep in REPRESENTATIONS:
    for source in ['BUSBRA','BREAST']:
        for seed in HEAD_SEEDS:
            h,st,hist=train_head(rep,source,seed)
            heads[(rep,source,seed)]=h
            status_rows.append(st)

status=pd.DataFrame(status_rows)
atomic_csv(
    status,
    OUT/'PHASE0_HEAD_TRAINING_STATUS.csv'
)
display(status.round(4))


## 9. Source-validation results

In [ ]:
source_rows=[]
source_scores={}

for rep in REPRESENTATIONS:
    for source in ['BUSBRA','BREAST']:
        X=source_units[(rep,source)]['X']
        y=source_units[(rep,source)]['y']
        _,va=splits[source]

        for seed in HEAD_SEEDS:
            s=head_scores(
                heads[(rep,source,seed)],
                X[va]
            )
            source_scores[(rep,source,seed)]=s
            auc,ap=auc_ap(y[va],s)
            source_rows.append({
                'representation':rep,
                'source':source,
                'seed':seed,
                'n_val':len(va),
                'auc':auc,
                'auprc':ap,
            })

source_result=pd.DataFrame(source_rows)
atomic_csv(
    source_result,
    OUT/'PHASE0_SOURCE_VALIDATION_RESULTS.csv'
)
display(source_result.round(4))


## 10. External evaluation

BUS-BRA target:
- image-level margins are averaged within patient.

BrEaST:
- case-level.

BUSI:
- image-level lesion cohort.


In [ ]:
def busbra_patient_scores(img_scores):
    t=busbra[['patient_key','y']].copy().reset_index(drop=True)
    t['score']=np.asarray(img_scores,dtype=float)
    return (
        t.groupby(['patient_key','y'],as_index=False)
        .agg(score=('score','mean'))
        .sort_values('patient_key')
        .reset_index(drop=True)
    )

target_labels={
    'BUSI':busi_all.y.to_numpy(dtype=int),
    'BREAST':breast.y.to_numpy(dtype=int),
}

external_scores={}
external_rows=[]

scenarios=[
    ('BUSBRA','BREAST'),
    ('BUSBRA','BUSI'),
    ('BREAST','BUSBRA'),
    ('BREAST','BUSI'),
]

for source,target in scenarios:
    for rep in REPRESENTATIONS:
        for seed in HEAD_SEEDS:
            h=heads[(rep,source,seed)]

            if target=='BUSBRA':
                img_s=head_scores(
                    h,features[rep]['BUSBRA']
                )
                pat=busbra_patient_scores(img_s)
                y=pat.y.to_numpy(dtype=int)
                s=pat.score.to_numpy(dtype=float)
                ids=pat.patient_key.astype(str).to_numpy()
            else:
                X=features[rep][target]
                y=target_labels[target]
                s=head_scores(h,X)
                ids=(
                    breast.patient_key.astype(str).to_numpy()
                    if target=='BREAST'
                    else busi_all.image_id.astype(str).to_numpy()
                )

            external_scores[(rep,source,target,seed)]={
                'ids':ids,
                'y':np.asarray(y,dtype=int),
                'score':np.asarray(s,dtype=float),
            }

            auc,ap=auc_ap(y,s)
            external_rows.append({
                'representation':rep,
                'source':source,
                'target':target,
                'seed':seed,
                'n':len(y),
                'auc':auc,
                'auprc':ap,
            })

external_result=pd.DataFrame(external_rows)
atomic_csv(
    external_result,
    OUT/'PHASE0_EXTERNAL_RESULTS.csv'
)
display(
    external_result.sort_values(
        ['source','target','representation','seed']
    ).round(4)
)


## 11. Primary matched URFM vs ImageNet-MAE paired bootstrap

Scenario effect is seed-balanced:

\[
\Delta =
\frac{1}{3}\sum_s(AUC_{URFM,s}-AUC_{MAE,s})
\]

The same pathology-stratified target resample is used for every model and seed.


In [ ]:
scenario_rows=[]
boot_detail=[]

for scen_idx,(source,target) in enumerate(scenarios):
    # Identity / labels must align across matched models and seeds.
    ref=external_scores[('URFM-224',source,target,HEAD_SEEDS[0])]
    ids=ref['ids']
    y=ref['y']

    for rep in PRIMARY_MODELS:
        for seed in HEAD_SEEDS:
            cur=external_scores[(rep,source,target,seed)]
            if not np.array_equal(ids,cur['ids']):
                raise RuntimeError(
                    f"Identity mismatch {rep}/{source}->{target}/seed{seed}"
                )
            if not np.array_equal(y,cur['y']):
                raise RuntimeError(
                    f"Label mismatch {rep}/{source}->{target}/seed{seed}"
                )

    # Point effect balanced over head seeds.
    seed_deltas=[]
    for seed in HEAD_SEEDS:
        u=external_scores[('URFM-224',source,target,seed)]['score']
        m=external_scores[('ImageNet-MAE-224',source,target,seed)]['score']
        seed_deltas.append(
            roc_auc_score(y,u)-roc_auc_score(y,m)
        )
    point=float(np.mean(seed_deltas))

    i0=np.flatnonzero(y==0)
    i1=np.flatnonzero(y==1)

    rng=np.random.default_rng(
        BOOTSTRAP_SEED+scen_idx
    )

    deltas=[]
    for b in range(BOOTSTRAP_REPS):
        s0=rng.choice(i0,size=len(i0),replace=True)
        s1=rng.choice(i1,size=len(i1),replace=True)
        idx=np.concatenate([s0,s1])
        yy=y[idx]

        ds=[]
        for seed in HEAD_SEEDS:
            u=external_scores[
                ('URFM-224',source,target,seed)
            ]['score'][idx]
            m=external_scores[
                ('ImageNet-MAE-224',source,target,seed)
            ]['score'][idx]
            ds.append(
                roc_auc_score(yy,u)-roc_auc_score(yy,m)
            )
        deltas.append(float(np.mean(ds)))

    deltas=np.asarray(deltas,dtype=float)

    scenario_rows.append({
        'source':source,
        'target':target,
        'urfm_minus_mae_auc_point':point,
        'seed42_delta':float(seed_deltas[0]),
        'seed43_delta':float(seed_deltas[1]),
        'seed44_delta':float(seed_deltas[2]),
        'bootstrap_median':float(np.median(deltas)),
        'ci_lo':float(np.quantile(deltas,.025)),
        'ci_hi':float(np.quantile(deltas,.975)),
        'p_urfm_better':float(np.mean(deltas>0)),
    })

    boot_detail.append(pd.DataFrame({
        'source':source,
        'target':target,
        'bootstrap_rep':np.arange(BOOTSTRAP_REPS),
        'urfm_minus_mae_auc':deltas,
    }))

scenario_effects=pd.DataFrame(scenario_rows)
bootstrap_detail=pd.concat(
    boot_detail,ignore_index=True
)

atomic_csv(
    scenario_effects,
    OUT/'PRIMARY_URFM_VS_MAE_EXTERNAL_EFFECTS.csv'
)
atomic_csv(
    bootstrap_detail,
    OUT/'PRIMARY_URFM_VS_MAE_BOOTSTRAP_3000.csv'
)

display(scenario_effects.round(4))


## 12. Source-validation matched effect

For each source:

\[
\Delta_{source}
=
\frac{1}{3}\sum_s
(AUC^{val}_{URFM,s}-AUC^{val}_{MAE,s})
\]

This is used only in the prespecified GO-B non-transfer test.


In [ ]:
source_effect_rows=[]

for source in ['BUSBRA','BREAST']:
    ds=[]
    for seed in HEAD_SEEDS:
        u=float(
            source_result[
                (source_result.representation=='URFM-224')
                & (source_result.source==source)
                & (source_result.seed==seed)
            ].auc.iloc[0]
        )
        m=float(
            source_result[
                (source_result.representation=='ImageNet-MAE-224')
                & (source_result.source==source)
                & (source_result.seed==seed)
            ].auc.iloc[0]
        )
        ds.append(u-m)

    source_effect_rows.append({
        'source':source,
        'urfm_minus_mae_source_val_auc':float(np.mean(ds)),
        'seed42_delta':float(ds[0]),
        'seed43_delta':float(ds[1]),
        'seed44_delta':float(ds[2]),
    })

source_effects=pd.DataFrame(source_effect_rows)
atomic_csv(
    source_effects,
    OUT/'PRIMARY_URFM_VS_MAE_SOURCE_EFFECTS.csv'
)
display(source_effects.round(4))


## 13. Secondary DINOv2 context

No GO/STOP criterion depends on DINOv2.


In [ ]:
context = (
    pd.concat(
        [
            source_result.assign(domain_role='source_validation'),
            external_result.assign(domain_role='external'),
        ],
        ignore_index=True,
        sort=False,
    )
)

atomic_csv(
    context,
    OUT/'ALL_MODEL_CONTEXT_RESULTS.csv'
)

summary = (
    external_result.groupby(
        ['representation','source','target'],
        as_index=False
    )
    .agg(
        mean_auc=('auc','mean'),
        sd_auc=('auc','std'),
        mean_auprc=('auprc','mean'),
    )
)

atomic_csv(
    summary,
    OUT/'EXTERNAL_MODEL_SUMMARY.csv'
)
display(summary.round(4))


## 14. Frozen Phase-0 decision

No post-hoc rescue path is available.


In [ ]:
# ---------------- GO-A ----------------
positive_scenarios=int(
    (scenario_effects.urfm_minus_mae_auc_point>0).sum()
)
median_external=float(
    scenario_effects.urfm_minus_mae_auc_point.median()
)
boot_supported=int(
    (
        scenario_effects.p_urfm_better
        >= GOA_BOOT_PROB
    ).sum()
)

go_a=(
    positive_scenarios>=GOA_MIN_POSITIVE_SCENARIOS
    and median_external>=GOA_MEDIAN_EFFECT
    and boot_supported>=GOA_MIN_BOOT_SUPPORTED
)

# ---------------- GO-B ----------------
source_map=source_effects.set_index(
    'source'
).urfm_minus_mae_source_val_auc.to_dict()

both_source_advantages=all(
    source_map[s]>=GOB_SOURCE_ADVANTAGE
    for s in ['BUSBRA','BREAST']
)

attenuation_rows=[]
for _,r in scenario_effects.iterrows():
    attenuation=float(
        source_map[r.source]
        - r.urfm_minus_mae_auc_point
    )
    attenuation_rows.append({
        'source':r.source,
        'target':r.target,
        'source_effect':float(source_map[r.source]),
        'external_effect':float(r.urfm_minus_mae_auc_point),
        'attenuation':attenuation,
    })

attenuation=pd.DataFrame(attenuation_rows)
atomic_csv(
    attenuation,
    OUT/'SOURCE_TO_EXTERNAL_ADVANTAGE_ATTENUATION.csv'
)

attenuated_count=int(
    (attenuation.attenuation>=GOB_ATTENUATION).sum()
)
external_boot_supported=int(
    (
        scenario_effects.p_urfm_better
        >= GOA_BOOT_PROB
    ).sum()
)

go_b=(
    both_source_advantages
    and median_external<=GOB_MAX_EXTERNAL_MEDIAN
    and attenuated_count>=GOB_MIN_ATTENUATED_SCENARIOS
    and external_boot_supported<=GOB_MAX_BOOT_SUPPORTED_EXTERNAL
)

if go_a:
    decision='GO_ULTRASOUND_PRETRAINING_EXTERNAL_ADVANTAGE'
elif go_b:
    decision='GO_SOURCE_ADVANTAGE_FAILS_TO_TRANSFER'
else:
    decision='STOP_FM_EXTERNAL_TRANSFER_DIRECTION'

decision_obj={
    'decision':decision,
    'GO_A':{
        'pass':bool(go_a),
        'positive_external_scenarios':positive_scenarios,
        'required_positive_scenarios':GOA_MIN_POSITIVE_SCENARIOS,
        'median_external_urfm_minus_mae_auc':median_external,
        'required_median_external_effect':GOA_MEDIAN_EFFECT,
        'bootstrap_supported_scenarios':boot_supported,
        'required_bootstrap_supported_scenarios':GOA_MIN_BOOT_SUPPORTED,
    },
    'GO_B':{
        'pass':bool(go_b),
        'source_effects':{
            k:float(v) for k,v in source_map.items()
        },
        'required_source_effect_each':GOB_SOURCE_ADVANTAGE,
        'median_external_effect':median_external,
        'maximum_external_median':GOB_MAX_EXTERNAL_MEDIAN,
        'attenuated_scenarios':attenuated_count,
        'required_attenuated_scenarios':GOB_MIN_ATTENUATED_SCENARIOS,
        'external_bootstrap_supported_scenarios':external_boot_supported,
        'maximum_supported_external_scenarios':GOB_MAX_BOOT_SUPPORTED_EXTERNAL,
    },
    'post_decision_rule':(
        'If STOP: do not launch LoRA/VPT/full fine-tuning or add another ultrasound FM '
        'to rescue the breast-US direction. If GO: design the full study before any new training.'
    ),
    'provenance_boundary':(
        'URFM named public breast pretraining sources do not list BUS-BRA, BrEaST, or BUSI; '
        'private-data overlap cannot be independently excluded.'
    ),
}

atomic_json(
    decision_obj,
    OUT/'PHASE0_FINAL_DECISION.json'
)

display(attenuation.round(4))
print(json.dumps(decision_obj,indent=2))


## 15. Package results

Return:

`FM_EXTERNAL_TRANSFER_PHASE0_RESULTS.zip`

Do not begin full fine-tuning or PEFT before reviewing this ZIP.


In [ ]:
result_zip=ROOT/'FM_EXTERNAL_TRANSFER_PHASE0_RESULTS.zip'

files_to_pack=[
    EXP/'FROZEN_PHASE0_CONFIG.json',
    EXP/'PRETRAINING_PROVENANCE_AUDIT.json',
    OUT/'URFM_LOAD_AUDIT.json',
    OUT/'MAE_LOAD_AUDIT.json',
    OUT/'BUSBRA_PHASE0_SOURCE_SPLIT.csv',
    OUT/'BREAST_PHASE0_SOURCE_SPLIT.csv',
    OUT/'PHASE0_HEAD_TRAINING_STATUS.csv',
    OUT/'PHASE0_SOURCE_VALIDATION_RESULTS.csv',
    OUT/'PHASE0_EXTERNAL_RESULTS.csv',
    OUT/'PRIMARY_URFM_VS_MAE_EXTERNAL_EFFECTS.csv',
    OUT/'PRIMARY_URFM_VS_MAE_BOOTSTRAP_3000.csv',
    OUT/'PRIMARY_URFM_VS_MAE_SOURCE_EFFECTS.csv',
    OUT/'SOURCE_TO_EXTERNAL_ADVANTAGE_ATTENUATION.csv',
    OUT/'EXTERNAL_MODEL_SUMMARY.csv',
    OUT/'ALL_MODEL_CONTEXT_RESULTS.csv',
    OUT/'PHASE0_FINAL_DECISION.json',
]

for rep in REPRESENTATIONS:
    for source in ['BUSBRA','BREAST']:
        for seed in HEAD_SEEDS:
            tag=f"{rep.replace('-','_')}_{source}_seed{seed}"
            files_to_pack += [
                OUT/f'{tag}_status.json',
                OUT/f'{tag}_history.csv',
            ]

with zipfile.ZipFile(
    result_zip,'w',zipfile.ZIP_DEFLATED
) as z:
    for p in files_to_pack:
        if not Path(p).exists():
            raise FileNotFoundError(p)
        z.write(p,arcname=Path(p).name)

print("Created:",result_zip)
print("Size MB:",result_zip.stat().st_size/1e6)
